In [0]:
# =============================================================
# JOB CODE — run_framework.py
# Medallion ETL : L0 (Bronze) / L1 (Silver) / L2 (Gold)
# =============================================================
# VIEW DESIGN:
#   - LOAD_TYPE = 'VIEW' rows are COMPLETELY SKIPPED here.
#   - Views are NEVER created by this notebook.
#   - After this job succeeds, Jenkins triggers view_pipeline.py
#     as a separate job (DBX_{GROUP_ID}_VIEW_JOB).
#   - This separation means: no duplicate view creation,
#     no DLT conflicts, clean single responsibility.
# =============================================================

In [0]:
dbutils.widgets.text("GROUP_ID",  "")
dbutils.widgets.text("RUN_LAYER", "ALL")

group_id  = dbutils.widgets.get("GROUP_ID").strip().upper()
run_layer = dbutils.widgets.get("RUN_LAYER").strip().upper()

if not group_id:
    raise Exception(
        "❌  GROUP_ID is empty.\n"
        "    Fix : Pass GROUP_ID when triggering the job.\n"
        "    Example : GROUP_ID = FINANCE_MEDALLION_L0"
    )

# Auto-detect layer from GROUP_ID suffix
if run_layer == "ALL":
    if   group_id.endswith("_L0"): run_layer = "L0"
    elif group_id.endswith("_L1"): run_layer = "L1"
    elif group_id.endswith("_L2"): run_layer = "L2"

print(f"{'═'*55}")
print(f"  GROUP_ID  : {group_id}")
print(f"  RUN_LAYER : {run_layer}")
print(f"  NOTE      : VIEW rows are skipped — handled by view_pipeline job")
print(f"{'═'*55}")

In [0]:
# ── Auto-detect catalog ───────────────────────────────────────
_PREFERRED = "demo_catalog"
try:
    available = [r[0] for r in spark.sql("SHOW CATALOGS").collect()]
    CATALOG = _PREFERRED if _PREFERRED in available else (
        "hive_metastore" if "hive_metastore" in available else
        (available[0] if available else _PREFERRED)
    )
    if CATALOG != _PREFERRED:
        print(f"⚠️  '{_PREFERRED}' not found → using '{CATALOG}'")
except Exception:
    CATALOG = "hive_metastore"

print(f"  CATALOG   : {CATALOG}\n")

In [0]:
import pandas as pd, requests, io, traceback
from datetime import datetime
from pyspark.sql import functions as F

In [0]:
# ── Helpers ───────────────────────────────────────────────────

def fmt_error(context, exc, query=None):
    lines = [
        f"",
        f"  ┌─ ERROR in {context} {'─'*max(0,44-len(context))}",
        f"  │  Type    : {type(exc).__name__}",
        f"  │  Message : {str(exc)[:300]}",
    ]
    if query:
        lines.append(f"  │  Query   : {query[:200]}")
    lines.append(f"  └{'─'*50}")
    return "\n".join(lines)


def read_source(source_url, file_format, delimiter=","):
    fmt = (file_format or "csv").strip().lower()
    url = (source_url or "").strip()
    if not url:
        raise ValueError("SOURCE_URL is empty in l0_detail.")
    print(f"   Reading  : {url[:80]}")
    is_http = url.lower().startswith("http")
    if is_http:
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        raw = resp.content
        if fmt == "csv":    return spark.createDataFrame(pd.read_csv(io.BytesIO(raw), sep=delimiter or ","))
        elif fmt == "json": return spark.createDataFrame(pd.read_json(io.BytesIO(raw)))
        elif fmt == "parquet": return spark.createDataFrame(pd.read_parquet(io.BytesIO(raw)))
        elif fmt in ("excel","xlsx"): return spark.createDataFrame(pd.read_excel(io.BytesIO(raw)))
        else: raise ValueError(f"Unsupported HTTP format '{fmt}'.")
    else:
        if fmt == "csv":     return spark.read.option("header","true").option("inferSchema","true").option("sep",delimiter or ",").csv(url)
        elif fmt == "json":  return spark.read.option("multiLine","true").json(url)
        elif fmt == "parquet": return spark.read.parquet(url)
        elif fmt == "delta": return spark.read.format("delta").load(url)
        else: raise ValueError(f"Unsupported cloud format '{fmt}'.")


def ensure_schema(schema):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")


def write_table(df, schema, table, load_type, merge_keys=None):
    ensure_schema(schema)
    full = f"{CATALOG}.{schema}.{table}"
    lt   = (load_type or "FULL").strip().upper()

    if lt == "FULL":
        df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(full)
    elif lt in ("INCREMENTAL","APPEND"):
        df.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(full)
    elif lt == "MERGE":
        if not merge_keys:
            raise ValueError(f"LOAD_TYPE=MERGE but MERGE_KEYS empty for {full}.")
        keys       = [k.strip() for k in merge_keys.split(",")]
        match_cond = " AND ".join([f"tgt.{k}=src.{k}" for k in keys])
        tmp_view   = f"_tmp_{full.replace('.','_')}"
        df.createOrReplaceTempView(tmp_view)
        spark.sql(f"CREATE TABLE IF NOT EXISTS {full} USING DELTA AS SELECT * FROM {tmp_view} WHERE 1=0")
        upd_cols = [c for c in df.columns if c not in keys]
        upd_set  = ", ".join([f"tgt.{c}=src.{c}" for c in upd_cols])
        ins_cols = ", ".join(df.columns)
        ins_vals = ", ".join([f"src.{c}" for c in df.columns])
        spark.sql(f"""
            MERGE INTO {full} AS tgt USING {tmp_view} AS src
            ON {match_cond}
            WHEN MATCHED THEN UPDATE SET {upd_set}
            WHEN NOT MATCHED THEN INSERT ({ins_cols}) VALUES ({ins_vals})
        """)
    else:
        print(f"   ⚠️  Unknown LOAD_TYPE '{lt}' — defaulting to FULL")
        df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(full)

    return spark.table(full).count()


def resolve_query(query, src_schema, src_table):
    q = query.strip()
    if src_schema and f"{src_schema}." in q and f"{CATALOG}.{src_schema}." not in q:
        q = q.replace(f"{src_schema}.", f"{CATALOG}.{src_schema}.")
    for kw in ("FROM ", "JOIN "):
        bare = f"{kw}{src_table}"
        full = f"{kw}{CATALOG}.{src_schema}.{src_table}"
        if src_table and bare in q and full not in q:
            q = q.replace(bare, full)
    return q


def write_audit(table_name, layer, status, rows, message, start_time, end_time):
    try:
        safe = str(message).replace("'","''")[:500]
        spark.sql(f"""
            INSERT INTO {CATALOG}.admin.audit_log
            VALUES ('{group_id}','{table_name}','{layer}','{status}',
                    {rows},'{safe}',
                    '{start_time.strftime('%Y-%m-%d %H:%M:%S')}',
                    '{end_time.strftime('%Y-%m-%d %H:%M:%S')}',
                    current_timestamp())
        """)
    except Exception as e:
        print(f"   ⚠️  Audit write failed: {e}")

In [0]:
# ── Validate header ───────────────────────────────────────────

try:
    hdr = spark.sql(f"""
        SELECT * FROM {CATALOG}.admin.data_flow_control_header
        WHERE DATA_FLOW_GROUP_ID='{group_id}' AND IS_ACTIVE='Y'
    """)
except Exception as e:
    raise RuntimeError(
        f"❌  Cannot read control_header.\n"
        f"    Detail : {e}\n"
        f"    Fix    : Run deployment pipeline first."
    )

if hdr.count() == 0:
    raise Exception(
        f"❌  GROUP_ID '{group_id}' not found or IS_ACTIVE != 'Y'.\n"
        f"    Query: SELECT * FROM {CATALOG}.admin.data_flow_control_header\n"
        f"           WHERE DATA_FLOW_GROUP_ID='{group_id}'"
    )

print(f"✅  Header found for {group_id}")

In [0]:
# ══════════════════════════════════════════════════════════
# L0 — BRONZE  (raw ingestion — VIEW rows skipped)
# ══════════════════════════════════════════════════════════

def run_l0():
    print(f"\n{'═'*55}\n  L0 — BRONZE INGESTION\n{'═'*55}")

    rows = spark.sql(f"""
        SELECT SOURCE_URL, SOURCE_OBJ_SCHEMA, SOURCE_OBJ_NAME,
               TARGET_SCHEMA, TARGET_TABLE, FILE_FORMAT,
               INPUT_FILE_FORMAT, LOAD_TYPE, DELIMETER
        FROM {CATALOG}.admin.data_flow_l0_detail
        WHERE DATA_FLOW_GROUP_ID='{group_id}' AND IS_ACTIVE='Y'
        AND   UPPER(LOAD_TYPE) != 'VIEW'
    """).collect()

    if not rows:
        raise Exception(
            f"❌  No active L0 rows for '{group_id}'.\n"
            f"    Fix: insert rows into data_flow_l0_detail."
        )

    print(f"  Sources : {len(rows)}")
    all_ok = True

    for row in rows:
        src_url   = row["SOURCE_URL"] or ""
        fmt       = row["INPUT_FILE_FORMAT"] or row["FILE_FORMAT"] or "csv"
        tgt_sch   = row["TARGET_SCHEMA"] or "bronze"
        tgt_tbl   = row["TARGET_TABLE"]  or ""
        load_type = row["LOAD_TYPE"]     or "FULL"
        delimiter = row["DELIMETER"]     or ","
        full_name = f"{CATALOG}.{tgt_sch}.{tgt_tbl}"

        if not tgt_tbl:
            print(f"   ⚠️  Skipping — TARGET_TABLE empty"); continue

        print(f"\n  ▶  {full_name}  |  {fmt}  |  {load_type}")
        start = datetime.now(); status = "FAILED"; rows_ct = 0; msg = ""

        try:
            df      = read_source(src_url, fmt, delimiter)
            df      = df.withColumn("_etl_group_id",F.lit(group_id)) \
                        .withColumn("_etl_layer",F.lit("L0")) \
                        .withColumn("_etl_load_ts",F.current_timestamp())
            rows_ct = write_table(df, tgt_sch, tgt_tbl, load_type)
            status  = "SUCCESS"; msg = f"{rows_ct:,} rows"
            print(f"   ✅ {msg} → {full_name}")
        except Exception as e:
            all_ok = False; msg = fmt_error(f"L0/{tgt_tbl}", e); print(msg)
        finally:
            end = datetime.now()
            print(f"   ⏱  {round((end-start).total_seconds(),1)}s")
            write_audit(tgt_tbl, "L0", status, rows_ct, msg, start, end)

    if not all_ok:
        raise Exception(
            f"❌  L0 FAILED.\n"
            f"    Check: SELECT * FROM {CATALOG}.admin.audit_log\n"
            f"           WHERE DATA_FLOW_GROUP_ID='{group_id}' AND ETL_LAYER='L0'\n"
            f"           ORDER BY LOAD_TS DESC"
        )

In [0]:
# ══════════════════════════════════════════════════════════
# L1 — SILVER  (VIEW rows skipped — handled by view_pipeline)
# ══════════════════════════════════════════════════════════

def run_l1():
    print(f"\n{'═'*55}\n  L1 — SILVER TRANSFORMATION\n{'═'*55}")

    rows = spark.sql(f"""
        SELECT SOURCE_OBJ_SCHEMA, SOURCE_OBJ_NAME,
               TARGET_SCHEMA, TARGET_TABLE, LOAD_TYPE,
               TRANSFORMATION_QUERY, MERGE_KEYS
        FROM {CATALOG}.admin.data_flow_l1_detail
        WHERE DATA_FLOW_GROUP_ID='{group_id}' AND IS_ACTIVE='Y'
        AND   UPPER(LOAD_TYPE) != 'VIEW'
    """).collect()

    if not rows:
        print(f"  ℹ️  No L1 TABLE rows — skipping Silver layer.")
        return

    print(f"  Transformations : {len(rows)}")
    all_ok = True

    for row in rows:
        src_sch    = row["SOURCE_OBJ_SCHEMA"] or "bronze"
        src_tbl    = row["SOURCE_OBJ_NAME"]   or ""
        tgt_sch    = row["TARGET_SCHEMA"]      or "silver"
        tgt_tbl    = row["TARGET_TABLE"]       or ""
        load_type  = row["LOAD_TYPE"]          or "MERGE"
        trans_q    = row["TRANSFORMATION_QUERY"] or ""
        merge_keys = row["MERGE_KEYS"]         or ""
        full_tgt   = f"{CATALOG}.{tgt_sch}.{tgt_tbl}"

        if not tgt_tbl: print(f"   ⚠️  Skipping — TARGET_TABLE empty"); continue

        print(f"\n  ▶  {CATALOG}.{src_sch}.{src_tbl} → {full_tgt}  |  {load_type}")
        start = datetime.now(); status = "FAILED"; rows_ct = 0; msg = ""

        try:
            q  = resolve_query(trans_q, src_sch, src_tbl) if trans_q else None
            df = spark.sql(q) if q else spark.table(f"{CATALOG}.{src_sch}.{src_tbl}")
            df = df.withColumn("_etl_group_id",F.lit(group_id)) \
                   .withColumn("_etl_layer",F.lit("L1")) \
                   .withColumn("_etl_load_ts",F.current_timestamp())
            rows_ct = write_table(df, tgt_sch, tgt_tbl, load_type, merge_keys)
            status  = "SUCCESS"; msg = f"{rows_ct:,} rows"
            print(f"   ✅ {msg} → {full_tgt}")
        except Exception as e:
            all_ok = False; msg = fmt_error(f"L1/{tgt_tbl}", e, trans_q); print(msg)
        finally:
            end = datetime.now()
            print(f"   ⏱  {round((end-start).total_seconds(),1)}s")
            write_audit(tgt_tbl, "L1", status, rows_ct, msg, start, end)

    if not all_ok:
        raise Exception(
            f"❌  L1 FAILED.\n"
            f"    Check: SELECT * FROM {CATALOG}.admin.audit_log\n"
            f"           WHERE DATA_FLOW_GROUP_ID='{group_id}' AND ETL_LAYER='L1'\n"
            f"           ORDER BY LOAD_TS DESC"
        )

In [0]:
# ══════════════════════════════════════════════════════════
# L2 — GOLD  (VIEW rows skipped — handled by view_pipeline)
# ══════════════════════════════════════════════════════════

def run_l2():
    print(f"\n{'═'*55}\n  L2 — GOLD AGGREGATION\n{'═'*55}")

    rows = spark.sql(f"""
        SELECT SOURCE_OBJ_SCHEMA, SOURCE_OBJ_NAME,
               TARGET_SCHEMA, TARGET_TABLE, LOAD_TYPE,
               TRANSFORMATION_QUERY, MERGE_KEYS
        FROM {CATALOG}.admin.data_flow_l2_detail
        WHERE DATA_FLOW_GROUP_ID='{group_id}' AND IS_ACTIVE='Y'
        AND   UPPER(LOAD_TYPE) != 'VIEW'
    """).collect()

    if not rows:
        print(f"  ℹ️  No L2 TABLE rows — skipping Gold layer.")
        return

    print(f"  Aggregations : {len(rows)}")
    all_ok = True

    for row in rows:
        src_sch    = row["SOURCE_OBJ_SCHEMA"] or "silver"
        src_tbl    = row["SOURCE_OBJ_NAME"]   or ""
        tgt_sch    = row["TARGET_SCHEMA"]      or "gold"
        tgt_tbl    = row["TARGET_TABLE"]       or ""
        load_type  = row["LOAD_TYPE"]          or "FULL"
        trans_q    = row["TRANSFORMATION_QUERY"] or ""
        merge_keys = row["MERGE_KEYS"]         or ""
        full_tgt   = f"{CATALOG}.{tgt_sch}.{tgt_tbl}"

        if not tgt_tbl: print(f"   ⚠️  Skipping — TARGET_TABLE empty"); continue

        print(f"\n  ▶  {CATALOG}.{src_sch}.{src_tbl} → {full_tgt}  |  {load_type}")
        start = datetime.now(); status = "FAILED"; rows_ct = 0; msg = ""

        try:
            q  = resolve_query(trans_q, src_sch, src_tbl) if trans_q else None
            df = spark.sql(q) if q else spark.table(f"{CATALOG}.{src_sch}.{src_tbl}")
            df = df.withColumn("_etl_group_id",F.lit(group_id)) \
                   .withColumn("_etl_layer",F.lit("L2")) \
                   .withColumn("_etl_load_ts",F.current_timestamp())
            rows_ct = write_table(df, tgt_sch, tgt_tbl, load_type, merge_keys)
            status  = "SUCCESS"; msg = f"{rows_ct:,} rows"
            print(f"   ✅ {msg} → {full_tgt}")
        except Exception as e:
            all_ok = False; msg = fmt_error(f"L2/{tgt_tbl}", e, trans_q); print(msg)
        finally:
            end = datetime.now()
            print(f"   ⏱  {round((end-start).total_seconds(),1)}s")
            write_audit(tgt_tbl, "L2", status, rows_ct, msg, start, end)

    if not all_ok:
        raise Exception(
            f"❌  L2 FAILED.\n"
            f"    Check: SELECT * FROM {CATALOG}.admin.audit_log\n"
            f"           WHERE DATA_FLOW_GROUP_ID='{group_id}' AND ETL_LAYER='L2'\n"
            f"           ORDER BY LOAD_TS DESC"
        )

In [0]:
# ── Execute ───────────────────────────────────────────────────

pipeline_start = datetime.now()

if   run_layer == "L0": run_l0()
elif run_layer == "L1": run_l1()
elif run_layer == "L2": run_l2()
else: run_l0(); run_l1(); run_l2()

In [0]:
total = round((datetime.now() - pipeline_start).total_seconds(), 1)
print(f"\n{'═'*55}")
print(f"  ✅  JOB COMPLETE — TABLE layers done")
print(f"  GROUP_ID  : {group_id}")
print(f"  LAYER     : {run_layer}")
print(f"  CATALOG   : {CATALOG}")
print(f"  DURATION  : {total}s")
print(f"{'─'*55}")
print(f"  VIEW rows (if any) will be handled by:")
print(f"  DBX_{group_id}_VIEW_JOB → view_pipeline.py")
print(f"{'═'*55}")